<a href="https://colab.research.google.com/github/jennifer-algabre/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
!git clone https://github.com/jennifer-algabre/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 47), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.86 MiB | 10.01 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [13]:
!find . -name "*.csv"

./outputs/refresh_queue_sample.csv
./data/raw/content_refresh_anonymized.csv


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The baseline rule prioritizes content pages that are visible in search results, have not been updated recently, have relatively low click-through rate, and rank outside the top search positions.

Each page receives a simple action score based on observable search performance signals. Pages with higher scores are prioritized for content review.

Reason codes:

- STALE_CONTENT – Page has not been updated for a long time.
- LOW_CTR – Click-through rate is below the chosen threshold.
- HIGH_VISIBILITY – Page still receives substantial impressions.
- LOW_RANK – Average search position suggests improvement opportunities.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --------------------------
# Baseline scoring rule
# --------------------------

df["action_score"] = 0
df["reason_code"] = ""

# Stale content
stale = df["days_since_last_update"] >= 180
df.loc[stale, "action_score"] += 2
df.loc[stale, "reason_code"] += "STALE_CONTENT;"

# Low CTR
low_ctr = df["ctr"] < 0.10
df.loc[low_ctr, "action_score"] += 2
df.loc[low_ctr, "reason_code"] += "LOW_CTR;"

# High visibility
high_impressions = df["impressions_90d"] >= 1000
df.loc[high_impressions, "action_score"] += 2
df.loc[high_impressions, "reason_code"] += "HIGH_VISIBILITY;"

# Lower search ranking
low_rank = df["avg_position"] > 10
df.loc[low_rank, "action_score"] += 1
df.loc[low_rank, "reason_code"] += "LOW_RANK;"

# Rank pages
baseline_queue = (
    df.sort_values(
        ["action_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

baseline_queue["priority_rank"] = baseline_queue.index + 1

import os
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print(output_path)

baseline_queue.head(20)

work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,action_score,reason_code,priority_rank
0,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.00,good,page_3_5,down,-89.1,7,STALE_CONTENT;LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,1
1,content_b16bd7307b39,client_7f2253d7e2,0.0,0.00,LOW,0.0,keyword article,informational,4329.0,27844.0,...,0.00,25.00,0.00,good,page_3_5,down,-69.7,7,STALE_CONTENT;LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,2
2,content_b28d1efd668f,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,transactional,6901.0,44938.0,...,3.83,6.57,0.96,excellent,page_3_5,stable,-17.2,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,3
3,content_813e88069237,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,commercial,4610.0,30146.0,...,1.37,10.37,0.52,excellent,page_3_5,down,-33.8,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,4
4,content_ff94c9b6b411,client_349c41201b,20.0,0.00,LOW,0.0,keyword article,informational,5375.0,35334.0,...,1.15,1.12,1.15,excellent,page_3_5,down,-32.4,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,5
5,content_66b4046cc144,client_7f2253d7e2,0.0,0.00,LOW,0.0,keyword article,informational,5502.0,36015.0,...,1.35,41.67,0.00,excellent,page_3_5,down,-89.3,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,6
6,content_a023517539fe,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,3530.0,22134.0,...,3.40,3.61,0.00,excellent,deep,up,27907.3,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,7
7,content_05e9b4cd9ccf,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,6974.0,44529.0,...,0.55,8.09,0.00,excellent,page_3_5,down,-42.1,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,8
8,content_0e70a832cb7a,client_4e07408562,10.0,0.00,LOW,0.0,keyword article,informational,3008.0,18398.0,...,1.71,2.31,0.00,excellent,page_3_5,up,24.8,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,9
9,content_8b36799b7e44,client_6208ef0f77,0.0,0.00,LOW,0.0,keyword article,informational,7394.0,47101.0,...,2.17,2.77,0.62,excellent,page_3_5,down,-62.7,5,LOW_CTR;HIGH_VISIBILITY;LOW_RANK;,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The highest-ranked pages are recommended for content refresh because they combine multiple observable signals:

- High search visibility
- Low click-through rate
- Older content
- Lower average ranking

Confidence is moderate because the recommendations are based only on observable search metrics rather than editorial review.

These recommendations could be wrong if:

- A page was intentionally not updated.
- Seasonal trends temporarily reduced performance.
- Business priorities favor other pages.
- External search algorithm changes affected rankings.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some pages may receive high action scores even though they are not the best candidates for content refresh. For example, pages with naturally low click-through rates or seasonal traffic patterns may be ranked highly by the baseline rule.

The baseline also cannot distinguish between pages affected by external events and pages that truly require updated content.

No product flags or future information were used when calculating the action score. The rule only relies on observable search performance metrics available before making the prioritization decision, reducing the risk of data leakage.